# Fine-tune POSTER on RAF-DB + Webcam Data

**Cần T4 GPU (>16GB VRAM).**

**Setup:** tạo folder `fer_finetune` trên Google Drive, upload:
| File | Source |
|---|---|
| `webcam_finetune_data.zip` | tools/prepare_finetune_data.py |
| `poster_best.pth` | outputs/models/ |
| `RAF-DB.zip` | data/DATASET/ (zip toàn bộ) |

(File `ir50.pth`, `mobilefacenet_model_best.pth.tar` không cần riêng — chúng đã nằm trong `poster_best.pth`.)

Sau đó sửa `DRIVE_FOLDER` ở cell dưới.

In [ ]:
DRIVE_FOLDER = '/content/drive/MyDrive/fer_finetune/colab_upload'
DRIVE_WEBZIP    = f'{DRIVE_FOLDER}/webcam_finetune_data.zip'
DRIVE_POSTERPT  = f'{DRIVE_FOLDER}/poster_best.pth'
DRIVE_RAFDB     = f'{DRIVE_FOLDER}/RAF-DB.zip'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, gc, time, math, zipfile
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Check files
for fp in [DRIVE_WEBZIP, DRIVE_POSTERPT, DRIVE_RAFDB]:
    ok = os.path.exists(fp)
    print(f'  {"[OK]" if ok else "[MISS]"} {os.path.basename(fp)}')
    if not ok:
        raise FileNotFoundError(f'Upload {os.path.basename(fp)} to {DRIVE_FOLDER}/')

In [ ]:
# Extract data
!unzip -q "{DRIVE_WEBZIP}" -d /content/webcam_data
!unzip -q "{DRIVE_RAFDB}" -d /content
print('Extracted!')

for cid in range(1, 8):
    n = len(os.listdir(f'/content/webcam_data/train/{cid}'))
    print(f'  Webcam class {cid}: {n}')

In [ ]:
# Find RAF-DB path
raf_root = '/content'
for candidate in ['/content/DATASET', '/content/train']:
    if os.path.exists(candidate):
        raf_root = os.path.dirname(candidate) if 'train' in candidate else candidate
        break
print(f'RAF-DB root: {raf_root}')

class RAFDataset(Dataset):
    def __init__(self, root, transform=None):
        self.paths, self.labels = [], []
        for cid in sorted(os.listdir(root)):
            cdir = os.path.join(root, cid)
            if not os.path.isdir(cdir): continue
            for fn in os.listdir(cdir):
                if fn.lower().endswith(('.jpg','.jpeg','.png')):
                    self.paths.append(os.path.join(cdir, fn))
                    self.labels.append(int(cid) - 1)
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

In [ ]:
# Copy model code from Drive (models/ir50.py, mobilefacenet.py, hyp_crossvit.py)
if not os.path.exists('/content/models'):
    !cp -r "{DRIVE_FOLDER}/models" /content/models
sys.path.insert(0, '/content')

# Import backbone classes
from models.ir50 import Backbone
from models.mobilefacenet import MobileFaceNet
from models.hyp_crossvit import HyVisionTransformer

# Define POSTER + SE_block inline (ensures exact match with checkpoint)
class SE_block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x)

class POSTER(nn.Module):
    def __init__(self, num_classes=7, depth=8):
        super().__init__()
        self.face_landback = MobileFaceNet([112, 112], 136)
        self.ir_back = Backbone(50, 0.0, 'ir')
        self.ir_layer = nn.Linear(1024, 512)
        self.pyramid_fuse = HyVisionTransformer(
            in_chans=49, q_chanel=49, embed_dim=512,
            depth=depth, num_heads=8, mlp_ratio=2.0,
            drop_rate=0.0, attn_drop_rate=0.0, drop_path_rate=0.1,
        )
        self.se_block = SE_block(512)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(512, num_classes)
    def forward(self, x):
        B = x.shape[0]
        x_face = F.interpolate(x, size=112)
        _, x_face = self.face_landback(x_face)
        x_face = x_face.view(B, -1, 49).transpose(1, 2)
        x_ir = self.ir_layer(self.ir_back(x))
        y = self.se_block(self.pyramid_fuse(x_ir, x_face))
        y = self.dropout(y)
        return self.head(y), y
print('Model classes imported + POSTER defined!')

In [ ]:
print('Building POSTER...')
model = POSTER(num_classes=7, depth=8).to(device)

# Load checkpoint
ckpt = torch.load(DRIVE_POSTERPT, map_location=device)
sd = ckpt.get('state_dict', ckpt)
miss, unexp = model.load_state_dict(sd, strict=False)
print(f'POSTER loaded: {len(miss)} missing, {len(unexp)} unexpected')

In [ ]:
BATCH_SIZE = 8

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),
    transforms.RandomApply([transforms.ColorJitter(0.3, 0.3, 0.3, 0.1)], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

raf_train = RAFDataset(f'{raf_root}/train', transform=train_tf)
raf_test  = RAFDataset(f'{raf_root}/test', transform=test_tf)
print(f'RAF-DB: {len(raf_train)} train / {len(raf_test)} test')

webcam_train = RAFDataset('/content/webcam_data/train', transform=train_tf)
print(f'Webcam: {len(webcam_train)}')

combined = ConcatDataset([raf_train] + [webcam_train] * 3)
train_loader = DataLoader(combined, BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
test_loader  = DataLoader(raf_test, BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train: {len(combined)} ({len(train_loader)} batches)')

In [ ]:
# SAM optimizer
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)
    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = torch.norm(torch.cat([p.grad.view(-1) for g in self.param_groups for p in g['params'] if p.grad is not None])).item()
        for g in self.param_groups:
            scale = g['rho'] / (grad_norm + 1e-12)
            for p in g['params']:
                if p.grad is None: continue
                self.state[p]['old_p'] = p.data.clone()
                p.add_(p.grad * scale)
        if zero_grad: self.zero_grad()
    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is None: continue
                p.data = self.state[p]['old_p']
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

criterion = nn.CrossEntropyLoss()
optimizer = SAM(model.parameters(), torch.optim.Adam, lr=2e-5, rho=0.05, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)

EPOCHS = 5  # giam xuong 5 cho nhanh, du voi it webcam data
best_acc = 0.0
print(f'Training: {EPOCHS} epochs, lr=2e-5, batch={BATCH_SIZE}')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    model.train()
    tloss, tcorr, tcnt = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits, _ = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.first_step(zero_grad=True)

        logits2, _ = model(imgs)
        criterion(logits2, labels).backward()
        optimizer.second_step(zero_grad=True)

        tloss += loss.item() * imgs.size(0)
        tcorr += (logits.argmax(1) == labels).sum().item()
        tcnt += imgs.size(0)

    model.eval()
    vcorr, vcnt, vloss = 0, 0, 0.0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits, _ = model(imgs)
            vloss += criterion(logits, labels).item() * imgs.size(0)
            vcorr += (logits.argmax(1) == labels).sum().item()
            vcnt += imgs.size(0)

    val_acc = vcorr / vcnt
    scheduler.step()
    print(f'Epoch {epoch:02d}/{EPOCHS} | {time.time()-t0:.1f}s | '
          f'Train: {tloss/tcnt:.4f}/{tcorr/tcnt:.4f} | Val: {vloss/vcnt:.4f}/{val_acc:.4f}')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'poster_best_finetuned.pth')
        print(f'  -> Saved (val_acc={best_acc:.4f})')

print(f'Done! Best: {best_acc:.4f}')

In [ ]:
# Save to Drive + download
!cp poster_best_finetuned.pth "{DRIVE_FOLDER}/"
print('Copied to Drive!')
from google.colab import files
files.download('poster_best_finetuned.pth')